# ArcNeuron on Google Colab

This notebook is only a thin runner for the repository. The neural architecture remains entirely in `arcneuron.py`; training, tuning, tokenization, and decoding remain in their corresponding Python files.

**Important:** the bundled `train.txt` and `tune.txt` are tiny research/demo corpora. They can prove that the pipeline learns and can be used for overfitting/recurrent-depth experiments, but they cannot produce a generally capable language model. Replace them with substantially more high-quality natural text before judging the architecture at scale.

The bundled corpus is currently Vietnamese, so the default generation prompt below is Vietnamese even though the notebook interface is English.

Enable a GPU from **Runtime → Change runtime type → GPU** before running.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = "https://github.com/ArcatureLabs/ArcNeuron.git"
ROOT = Path("/content/ArcNeuron")

if not (ROOT / "arcneuron.py").is_file():
    if ROOT.exists():
        subprocess.run(["rm", "-rf", str(ROOT)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)

def run_live(command):
    """Run a child process and stream every output line into the Colab cell."""
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, command)

print("working directory:", Path.cwd())


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Select Runtime > Change runtime type > GPU and run again.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")


## Experiment configuration

These defaults are no longer a 300-step smoke test. They are still small enough for Colab, but long enough to let the bundled corpus visibly overfit and to make generation less random. For a real model, enlarge the corpus before enlarging the model.


In [ ]:
TRAIN_STEPS = 3000
TUNE_STEPS = 600

VOCAB_SIZE = 1024
BATCH_SIZE = 8
GRAD_ACCUM = 1
CONTEXT = 256

DIM = 384
HEADS = 6
KV_HEADS = 2
FFN_DIM = 1024
PRELUDE_LAYERS = 1
CORE_LAYERS = 2
CODA_LAYERS = 1
MAX_DEPTH = 4

BASE_CKPT = "arcneuron.pt"
TUNED_CKPT = "arcneuron-tuned.pt"


## Train the base model

Loss is printed live. The child Python process runs unbuffered, so you should see progress while the GPU is training instead of only after the command exits.


In [ ]:
train_command = [
    sys.executable, "-u", "train.py",
    "--data", "train.txt",
    "--out", BASE_CKPT,
    "--steps", str(TRAIN_STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--context", str(CONTEXT),
    "--vocab-size", str(VOCAB_SIZE),
    "--dim", str(DIM),
    "--heads", str(HEADS),
    "--kv-heads", str(KV_HEADS),
    "--ffn-dim", str(FFN_DIM),
    "--prelude-layers", str(PRELUDE_LAYERS),
    "--core-layers", str(CORE_LAYERS),
    "--coda-layers", str(CODA_LAYERS),
    "--max-depth", str(MAX_DEPTH),
    "--warmup", "100",
    "--log-every", "5",
    "--eval-every", "100",
    "--eval-batches", "6",
    "--save-every", "500",
]

run_live(train_command)


## Tune response behavior

`tune.txt` now contains direct natural questions followed immediately by strong natural answers. It no longer teaches the model to say meta-sentences such as “a good answer should...”. Tuning still updates the exact same ArcNeuron weights with the exact same next-token loss.


In [ ]:
tune_command = [
    sys.executable, "-u", "tune.py",
    "--checkpoint", BASE_CKPT,
    "--data", "tune.txt",
    "--replay-data", "train.txt",
    "--out", TUNED_CKPT,
    "--steps", str(TUNE_STEPS),
    "--batch-size", str(max(1, BATCH_SIZE // 2)),
    "--context", str(CONTEXT),
    "--max-depth", str(MAX_DEPTH),
    "--replay-ratio", "0.20",
    "--log-every", "5",
    "--save-every", "100",
]

run_live(tune_command)


## Generate

Generation returns only the continuation by default. Generic nucleus/top-k sampling and a mild repetition penalty reduce obvious looping; they do not contain any knowledge or reasoning rules.


In [ ]:
from generate import load_model, generate

device = torch.device("cuda")
checkpoint = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
model, tokenizer = load_model(checkpoint, device)

PROMPT = "Một con mèo bị mất một chân có còn là động vật có vú không? Giải thích."
DEPTH = 4

answer = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    depth=DEPTH,
    max_new_tokens=128,
    temperature=0.70,
    top_k=40,
    top_p=0.90,
    repetition_penalty=1.08,
    repeat_window=96,
    include_prompt=False,
    device=device,
)

print("Prompt:", PROMPT)
print("Answer:", answer)


## Compare recurrent depth

Use the same checkpoint and prompt while changing only recurrence depth. Greedy decoding makes the comparison deterministic. More depth is not assumed to be better; this cell exists to measure that claim.


In [ ]:
for depth in [1, 2, 4, 8]:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    answer = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=PROMPT,
        depth=depth,
        max_new_tokens=128,
        temperature=0.0,
        top_k=0,
        top_p=1.0,
        repetition_penalty=1.06,
        repeat_window=96,
        include_prompt=False,
        device=device,
    )

    print(f"\n{'=' * 24} depth={depth} {'=' * 24}\n")
    print(answer)


## Download the checkpoint from Colab

Run this before the runtime resets if you want to keep the trained checkpoint locally.


In [ ]:
from google.colab import files

path = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
files.download(path)
